# Day 12 · Exercise 3: filtered_search

**What you'll build:** `search_with_filter(collection: chromadb.Collection, query_embedding: list[float], where: dict, top_k: int) -> list[dict]` — a function that runs a Chroma vector search restricted to only the documents that pass a `where`-clause metadata filter before any similarity ranking occurs.

**Why it matters:** Combining a metadata filter with a vector search is the core pattern that makes a vector database useful in production — you stop scanning documents that could never belong in the answer, and every result you get back satisfies both a structured condition and semantic relevance.

## Your Implementation

In [ ]:
import chromadb

def search_with_filter(
    collection: chromadb.Collection,
    query_embedding: list[float],
    where: dict,
    top_k: int,
) -> list[dict]:
    """Run a vector search on `collection` restricted to documents matching `where`.

    The `where` dict is forwarded directly to Chroma's `collection.query` call.
    Supported operators include `$eq`, `$ne`, `$in`, `$nin`, `$gt`, `$gte`,
    `$lt`, `$lte`, and compound `$and` / `$or`.

    Args:
        collection: A Chroma Collection that already contains embedded documents.
        query_embedding: The embedding vector for the query (list of floats).
        where: A Chroma where-clause dict, e.g.
            ``{"category": {"$in": ["electronics", "books"]}}`` or
            ``{"$and": [{"rating": {"$gte": 4}}, {"in_stock": {"$eq": 1}}]}``.
        top_k: Maximum number of results to return.  Chroma may return fewer if
            the filter leaves fewer than `top_k` documents in the candidate pool.

    Returns:
        A list of dicts, one per result, each with keys:
            - ``"id"``       (str)   — the document's Chroma ID
            - ``"text"``     (str)   — the document text
            - ``"distance"`` (float) — the embedding distance (lower = closer)
            - ``"metadata"`` (dict)  — the document's metadata dict

    Example:
        >>> results = search_with_filter(
        ...     collection=col,
        ...     query_embedding=embed("wireless headphones"),
        ...     where={"rating": {"$gte": 4}},
        ...     top_k=3,
        ... )
        >>> results[0]["metadata"]["rating"]
        4.5
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
import chromadb

_PASS, _FAIL = '✅', '❌'

# ── Build a small in-memory collection used by all checks ──────────────────
_client = chromadb.EphemeralClient()
# Delete if it already exists (re-run safety)
try:
    _client.delete_collection("check_products")
except Exception:
    pass

_col = _client.create_collection("check_products")

_products = [
    {"id": "p0",  "text": "Noise-cancelling over-ear headphones",          "category": "electronics", "rating": 4.5, "in_stock": 1},
    {"id": "p1",  "text": "Bluetooth mechanical keyboard compact layout",   "category": "electronics", "rating": 3.8, "in_stock": 1},
    {"id": "p2",  "text": "Ergonomic office chair lumbar support",         "category": "furniture",   "rating": 4.2, "in_stock": 0},
    {"id": "p3",  "text": "Python crash course programming book",          "category": "books",       "rating": 4.7, "in_stock": 1},
    {"id": "p4",  "text": "Ceramic pour-over coffee dripper",             "category": "kitchen",     "rating": 3.5, "in_stock": 1},
    {"id": "p5",  "text": "Standing desk adjustable height motorised",    "category": "furniture",   "rating": 4.6, "in_stock": 0},
    {"id": "p6",  "text": "Deep learning textbook neural networks",       "category": "books",       "rating": 4.9, "in_stock": 1},
    {"id": "p7",  "text": "USB-C hub seven ports fast-charge",            "category": "electronics", "rating": 3.9, "in_stock": 1},
    {"id": "p8",  "text": "Cast-iron skillet pre-seasoned 10 inch",       "category": "kitchen",     "rating": 4.8, "in_stock": 1},
    {"id": "p9",  "text": "Wireless gaming mouse high DPI adjustable",    "category": "electronics", "rating": 4.1, "in_stock": 0},
]

# Use tiny deterministic fake embeddings (dimension 4) — no Ollama needed
import math

def _fake_embed(text: str) -> list[float]:
    """Deterministic 4-d embedding from text length and first char code."""
    n = len(text)
    c = ord(text[0]) if text else 65
    v = [math.sin(n * 0.1), math.cos(c * 0.05), math.sin(c * 0.1 + n * 0.05), math.cos(n * 0.07)]
    norm = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / norm for x in v]

_col.add(
    ids=[p["id"] for p in _products],
    embeddings=[_fake_embed(p["text"]) for p in _products],
    documents=[p["text"] for p in _products],
    metadatas=[{"category": p["category"], "rating": p["rating"], "in_stock": p["in_stock"]} for p in _products],
)

_query_vec = _fake_embed("computing peripherals and gadgets")


def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and returns a list
    try:
        assert callable(search_with_filter), "search_with_filter is not defined or not callable"
        result = search_with_filter(
            collection=_col,
            query_embedding=_query_vec,
            where={"category": {"$eq": "electronics"}},
            top_k=3,
        )
        assert isinstance(result, list), f"expected list, got {type(result).__name__}"
        print(f"{_PASS} Check 1/{total}: function exists, returns a list")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1/{total}: {e}")
        return

    # Check 2: each item is a dict with required keys
    try:
        required_keys = {"id", "text", "distance", "metadata"}
        for i, item in enumerate(result):
            missing = required_keys - set(item.keys())
            assert not missing, f"result[{i}] missing keys: {missing}"
        print(f"{_PASS} Check 2/{total}: each result dict has keys id, text, distance, metadata")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 2/{total}: {e}")

    # Check 3: $gte filter — all returned items have rating >= 4.0
    try:
        high_rated = search_with_filter(
            collection=_col,
            query_embedding=_query_vec,
            where={"rating": {"$gte": 4.0}},
            top_k=5,
        )
        assert len(high_rated) > 0, "$gte filter returned no results"
        bad = [r for r in high_rated if r["metadata"]["rating"] < 4.0]
        assert not bad, f"{len(bad)} result(s) violate rating >= 4.0: {[b['metadata']['rating'] for b in bad]}"
        print(f"{_PASS} Check 3/{total}: $gte filter — all {len(high_rated)} results have rating >= 4.0")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 3/{total}: {e}")

    # Check 4: $and compound filter — all results satisfy both conditions
    try:
        compound = search_with_filter(
            collection=_col,
            query_embedding=_query_vec,
            where={
                "$and": [
                    {"in_stock": {"$eq": 1}},
                    {"rating":   {"$gte": 4.0}},
                ]
            },
            top_k=5,
        )
        assert len(compound) > 0, "$and compound filter returned no results"
        bad_stock  = [r for r in compound if r["metadata"]["in_stock"] != 1]
        bad_rating = [r for r in compound if r["metadata"]["rating"] < 4.0]
        assert not bad_stock,  f"{len(bad_stock)} result(s) not in stock"
        assert not bad_rating, f"{len(bad_rating)} result(s) have rating < 4.0"
        print(f"{_PASS} Check 4/{total}: $and filter — all {len(compound)} results are in-stock AND rated >= 4.0")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 4/{total}: {e}")

    print()
    if score == total:
        print("🎉 Exercise complete!")
    print(f"  {score}/{total} passed." + (" Keep going!" if score < total else ""))

_run_checks()

## Bonus Challenge

Right now `search_with_filter` accepts a pre-computed `query_embedding`. In a real application the caller usually has a raw text query, not a vector.

Extend your design: write a thin wrapper `text_search_with_filter(collection, query_text, embed_fn, where, top_k)` that calls `embed_fn(query_text)` to get the vector, then delegates to `search_with_filter`. This separation — embedding vs. retrieval — will matter on Day 15 when you swap embedding models without touching the retrieval logic. Hint: this is the **strategy pattern** applied to embeddings.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import chromadb

def search_with_filter(
    collection: chromadb.Collection,
    query_embedding: list[float],
    where: dict,
    top_k: int,
) -> list[dict]:
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where,
        include=["documents", "distances", "metadatas"],
    )
    output = []
    ids        = results["ids"][0]
    documents  = results["documents"][0]
    distances  = results["distances"][0]
    metadatas  = results["metadatas"][0]
    for doc_id, text, dist, meta in zip(ids, documents, distances, metadatas):
        output.append({
            "id":       doc_id,
            "text":     text,
            "distance": dist,
            "metadata": meta,
        })
    return output
```

**Why this works:** `collection.query` accepts a `where` dict and applies it as a pre-filter before computing any similarity scores — so passing the dict straight through is both correct and sufficient. The result object uses a `[0]` outer list because Chroma supports batched queries; since we always send one query embedding we always index into index 0. Zipping the four parallel lists into dicts gives the caller a clean, self-contained record for each hit.
</details>